In [ ]:
#| hide
#| eval: false
! [ -e /content ] && pip install -Uqq xcube #upgrade fastai on colab

In [ ]:
#| default_exp utils

In [62]:
#| export
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from fastcore.all import *
# from xcube.imports import *
from xcube.torch_imports import *
from xcube.fastai_imports import *
import logging
from tabulate import tabulate
import torch
import torch.nn as nn
import numpy as np
import math

In [ ]:
#| hide
from nbdev.showdoc import *
%load_ext autoreload
%autoreload 2

# utils

> Utilities needed for little repititive tasks

In [ ]:
#| export
def namestr(obj, namespace=None):
    "Returns the name of the object `obj` passed"
    return [name for name in namespace if namespace[name] is obj]

Here's an example of how `namestr` works:

In [ ]:
a = 'some_var'
test_eq(namestr(a, globals()), ['a'])

In [ ]:
#| export
def list_files(startpath):
    """ [simulates the linux tree cmd] 
    (https://stackoverflow.com/questions/9727673/list-directory-tree-structure-in-python)
    """ 
    
    if isinstance(startpath, Path): startpath = str(startpath) 
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print('{}{}/'.format(indent, os.path.basename(root)))
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print('{}{}'.format(subindent, f))

In [ ]:
#| export
def make_paths(path, prefix=None):
    """
    with `path` as basedir, makes data and models dir and 
    returns a dictionary of relevant pathlib objects
    """
    path_data = path/'data'
    path_model = path/'models'

    path_model.mkdir(exist_ok=True)
    path_data.mkdir(exist_ok=True)
    (path_model/'collab').mkdir(exist_ok=True)

    data = path_data/(prefix+'.csv')
    dls_lm_path, dls_lm_r_path = path_model/f"{prefix}_dls_lm.pkl", path_model/f"{prefix}_dls_lm_r.pkl"
    dls_lm_vocab_path, dls_lm_vocab_r_path = path_model/f"{prefix}_dls_lm_vocab.pkl", path_model/f"{prefix}_dls_lm_vocab_r.pkl"
    lm_path, lm_r_path = path_model/f"{prefix}_lm.pth", path_model/f"{prefix}_lm_r.pth"
    lm_finetuned_path, lm_finetuned_r_path = path_model/f"{prefix}_lm_finetuned.pth", path_model/f"{prefix}_lm_finetuned_r.pth"
    dsets_clas_path, dsets_clas_r_path = path_model/f"{prefix}_dset_clas.pkl", path_model/f"{prefix}_dset_clas_r.pkl"
    dls_clas_path, dls_clas_r_path = path_model/f"{prefix}_dls_clas.pkl", path_model/f"{prefix}_dls_clas_r.pkl"
    clas_path, clas_r_path = path_model/f"{prefix}_clas.pth", path_model/f"{prefix}_clas_r.pth"
    collab_bootst_path = path_model/f"{prefix}_tok_lbl_info.pkl"
    collab_data_path = path_data/f"{prefix}_tok_lbl.ft"
    collab_tok_path = path_data/f"{prefix}_tok.ft"
    collab_lbl_path = path_data/f"{prefix}_lbl.ft"
    dls_collab_path = path_model/f"{prefix}_dls_collab.pkl"
    dls_learn_rank_path = path_model/f"{prefix}_dls_learn_rank.pkl"
    collab_path = path_model/'collab'/f"{prefix}_collab.pth"
    plist = [path, path_data, path_model, 
             data, 
             dls_lm_path, dls_lm_r_path,
             dls_lm_vocab_path, dls_lm_vocab_r_path,
             lm_path, lm_r_path,
             lm_finetuned_path, lm_finetuned_r_path,
             dsets_clas_path, dsets_clas_r_path,
             dls_clas_path, dls_clas_r_path,
             clas_path, clas_r_path,
             collab_bootst_path,
             collab_data_path,
             collab_tok_path,
             collab_lbl_path,
             dls_collab_path,
             dls_learn_rank_path,
             collab_path]
    pdir = {}
    for o in plist:  pdir[namestr(o, locals())[0]] = o
    return pdir

In [ ]:
with tempfile.TemporaryDirectory() as tempdirname:
    print(f"created temporary dir: {tempdirname}")
    _paths = make_paths(Path(tempdirname), "mimic3-9k")
    for v in _paths.values(): v.touch()
    list_files(tempdirname)

created temporary dir: /tmp/tmpi1evi3rs
tmpi1evi3rs/
    data/
        mimic3-9k_tok.ft
        mimic3-9k_lbl.ft
        mimic3-9k.csv
        mimic3-9k_tok_lbl.ft
    models/
        mimic3-9k_dls_clas.pkl
        mimic3-9k_dls_lm.pkl
        mimic3-9k_lm_r.pth
        mimic3-9k_lm_finetuned_r.pth
        mimic3-9k_tok_lbl_info.pkl
        mimic3-9k_dls_lm_vocab_r.pkl
        mimic3-9k_dls_collab.pkl
        mimic3-9k_clas.pth
        mimic3-9k_dset_clas.pkl
        mimic3-9k_dls_lm_vocab.pkl
        mimic3-9k_dls_learn_rank.pkl
        mimic3-9k_dls_lm_r.pkl
        mimic3-9k_dset_clas_r.pkl
        mimic3-9k_lm.pth
        mimic3-9k_clas_r.pth
        mimic3-9k_lm_finetuned.pth
        mimic3-9k_dls_clas_r.pkl
        collab/
            mimic3-9k_collab.pth


In [ ]:
#| export
def plot_hist(data, x_label=None, y_label=None, title="Histogram"):
    n, bins, pathches = plt.hist(data)
    plt.grid(axis='y', color='b')
    # plt.yscale('log')
    if x_label is not None: plt.xlabel(x_label)
    if y_label is not None: plt.ylabel(y_label)
    maxfreq = n.max()
    plt.ylim(ymax=np.ceil(maxfreq / 10) * 10 if maxfreq % 10 else maxfreq + 10)
    plt.title(title);

In [ ]:
#| export
def plot_reduction(X, tSNE=True, n_comps=None, perplexity=30, figsize=(6,4)):
    """
    PCA on X and plots the first two principal components, returns the decomposition 
    and the explained variances for each directions,
    if `tSNE` then does a tSNE after PCA.
    """
    reduction = "tSNE" if tSNE else "PCA"
    pca = PCA(n_components=n_comps, svd_solver="full")
    X_red = pca.fit_transform(X)
    if tSNE:
        tsne = TSNE(n_components=2, perplexity=perplexity)
        X_red = tsne.fit_transform(X_red[:, :50])
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(1,1,1)
    plt.scatter(X_red[:, 0], X_red[:, 1], marker='x')
    ax.set_xlabel("1st component")
    ax.set_ylabel("2nd component")
    ax.set_title(f"{reduction} Decomposition")
    plt.show()
    return X_red, pca.explained_variance_ratio_

In [ ]:
#| export
def test_eqs(*args):
    for i in range(len(args)-1):
        test_eq(args[i], args[i+1])

In [ ]:
test_eqs(1, 1, 9//9)

In [ ]:
#| export
from fastai.callback.tracker import SaveModelCallback

In [ ]:
#| export
def validate(learner, cb=SaveModelCallback, **kwargs):
    "validates a `learner` within a context manager after temporarily removing `cb` if it exists"
    save_cb_idx = learner.cbs.argfirst(lambda o: isinstance(o, cb))
    if save_cb_idx is None:
        print(learner.validate(**kwargs))
        return
    print(f'best so far = {learner.cbs[save_cb_idx].best}')
    with learner.removed_cbs(learner.cbs[save_cb_idx]):
        vals = learner.validate(**kwargs)
        names = learner.recorder._valid_mets.map(Self.name())
        print('\n'.join([f"{n} = {v}" for n,v in zip(names,vals)]))
    save_cb_idx = learner.cbs.argfirst(lambda o: isinstance(o, cb))
    print(f'best so far = {learner.cbs[save_cb_idx].best}')
    return vals

In [ ]:
#| export
def cudamem(device=default_device()):
    using = torch.cuda.max_memory_reserved(device)/1024**3
    total = torch.cuda.get_device_properties(device).total_memory/1024**3
    print(f"GPU: {torch.cuda.get_device_name(default_device())}")
    print(f"You are using {using} GB")
    print(f"Total GPU memory = {total} GB")

In [ ]:
#| export
import requests
from bs4 import BeautifulSoup
def get_description(codes):
    "descriptions of ICD10 codes"
    # Initialize an empty dictionary to store code descriptions
    icd10_descriptions = {}
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.36"
    }
    with requests.Session() as session:
        for code in codes:
            url_cm = f"https://icd10coded.com/cm/{code}"
            url_pcs = f"https://icd10coded.com/pcs/{code}"
            
            for url in (url_pcs, url_cm):   
                with session.get(url, headers=headers, timeout=5) as response:
                    if response.status_code == 200:
                        soup = BeautifulSoup(response.content, 'html.parser')
                        description_element = soup.find('span', {'class': 'lead'})
                        if description_element:
                            description = description_element.text.strip()
                            icd10_descriptions[code] = description
                            break
                        else:
                            icd10_descriptions[code] = "Description not found"
                    else:
                        # print(f"An error occurred.")
                        icd10_descriptions[code] = "Code not found"
    return icd10_descriptions

In [ ]:
#| export
import sys
from contextlib import contextmanager
@contextmanager
def suppress_stdout():
    with open(os.devnull, 'w') as devnull:
        old_stdout = sys.stdout
        sys.stdout = devnull
        try:
            yield
        finally:
            sys.stdout = old_stdout

In [ ]:
# Example function with print statements
def some_function():
    print("This will be printed")
    print("So will this")

# Calling the function normally
print("Before suppressing print")
some_function()
print("After suppressing print")

# Calling the function with suppressed print
print("\nBefore suppressing print inside the function")
with suppress_stdout():
    some_function()
print("After suppressing print inside the function")

Before suppressing print
This will be printed
So will this
After suppressing print

Before suppressing print inside the function
After suppressing print inside the function


In [ ]:
#| export
def is_colab():
    # Check if the 'google.colab' module is available
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

    # Check for specific environment variables typically set in Colab
    if 'COLAB_GPU' in os.environ:
        IN_COLAB = True

    if IN_COLAB:
        print("Running in Google Colab")
    else:
        print("Not running in Google Colab")

    if IN_COLAB:
        try:
            subprocess.check_call(["pip", "install", 'GitPython'])
            print(f"Successfully installed GitPython")
            from git import Repo
            repo = Repo.clone_from('https://github.com/debjyotiSRoy/xcube.git', Path.cwd().parent/'xcube')
            repo.git.checkout('plant')
        except subprocess.CalledProcessError as e:
            print(f"Failed to install GitPython: {e}")
    return IN_COLAB

In [ ]:
#| export
def setup_logging(log_file):
    # Configure logging with the Formatter
    format_str = "[%(asctime)s] %(levelname)s [%(module)s.%(funcName)s:%(lineno)d] %(message)s"
    logging.basicConfig(level=logging.INFO, handlers=[logging.StreamHandler()], format=format_str)
    # Configure logging to write to a log file
    file_handler = logging.FileHandler(log_file)
    # file_handler.setFormatter(formatter)
    logging.getLogger().addHandler(file_handler)

## Model Utilities

In [58]:
#| export
def compute_trainable_params1(model, recursive_depth=0):
    """
    Compute the number of trainable and total parameters in the model.
    If recursive_depth > 0, generate a table of parameters for child modules up to the specified depth,
    including a Level # column and a totals row. If recursive_depth = np.inf, include all modules
    regardless of depth.
    
    Args:
        model (nn.Module): The PyTorch model to analyze.
        recursive_depth (int or float): Depth of recursion for child modules. Default is 0 (no recursion).
            Use np.inf to include all modules at any depth.
    
    Returns:
        tuple: (trainable_params, total_params)
    """
    # Compute total trainable and all parameters for the entire model
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_percentage = 100 * trainable_params / total_params if total_params > 0 else 0.0

    # Log the summary for the entire model
    logging.info(
        f"📊 trainable params: {trainable_params:,} || all params: {total_params:,} || trainable%: {trainable_percentage:.4f}"
    )

    # If recursive_depth > 0, generate a table for child modules
    if recursive_depth > 0:
        table_data = []
        param_ids = set()  # Track parameter IDs to avoid double-counting
        total_trainable_in_table = 0
        total_params_in_table = 0

        for name, module in model.named_modules():
            # Compute module depth
            depth = name.count('.') + 1 if name else 0
            # Skip depth check if recursive_depth is infinite
            if recursive_depth != np.inf and depth > recursive_depth:
                continue

            # Compute parameters for the current module
            module_trainable = 0
            module_total = 0
            module_param_ids = set()

            # Only count parameters directly registered in the current module
            for _, p in module.named_parameters(recurse=False):
                p_id = id(p)
                if p_id not in param_ids:
                    module_param_ids.add(p_id)
                    module_total += p.numel()
                    if p.requires_grad:
                        module_trainable += p.numel()

            if module_total > 0:
                module_percentage = 100 * module_trainable / module_total if module_total > 0 else 0.0
                indent = "  " * depth
                table_data.append([
                    depth,
                    indent + (name or "root"),
                    f"{module_trainable:,}",
                    f"{module_total:,}",
                    f"{module_percentage:.4f}%"
                ])
                param_ids.update(module_param_ids)
                total_trainable_in_table += module_trainable
                total_params_in_table += module_total

        # Add totals row
        total_percentage = 100 * total_trainable_in_table / total_params_in_table if total_params_in_table > 0 else 0.0
        table_data.append([
            "-",  # No level for totals
            "Total",
            f"{total_trainable_in_table:,}",
            f"{total_params_in_table:,}",
            f"{total_percentage:.4f}%"
        ])

        # Create and log the table
        if table_data:
            headers = ["🔢 Level #", "🧩 Module", "📏 Trainable Params", "📊 Total Params", "📈 Trainable %"]
            table = tabulate(
                table_data,
                headers=headers,
                tablefmt="fancy_grid",
                stralign="left",
                numalign="right",
                floatfmt=".4f"
            )
            logging.info(f"\nParameter breakdown by module (depth <= {recursive_depth}):\n{table}")

    return trainable_params, total_params

def compute_trainable_params2(model, recursive_depth=0):
    """
    Compute the number of trainable and total parameters in the model.
    If recursive_depth > 0, generate a table of parameters for child modules up to the specified depth,
    including a Level # column and a totals row. If recursive_depth = np.inf, include all modules
    regardless of depth. nn.Parameter objects are treated as pseudo-modules and listed separately.
    
    Args:
        model (nn.Module): The PyTorch model to analyze.
        recursive_depth (int or float): Depth of recursion for child modules. Default is 0 (no recursion).
            Use np.inf to include all modules at any depth.
    
    Returns:
        tuple: (trainable_params, total_params)
    """
    # Compute total trainable and all parameters for the entire model
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_percentage = 100 * trainable_params / total_params if total_params > 0 else 0.0

    # Log the summary for the entire model
    logging.info(
        f"📊 trainable params: {trainable_params:,} || all params: {total_params:,} || trainable%: {trainable_percentage:.4f}"
    )

    # If recursive_depth > 0, generate a table for child modules and parameters
    if recursive_depth > 0:
        table_data = []
        param_ids = set()  # Track parameter IDs to avoid double-counting
        total_trainable_in_table = 0
        total_params_in_table = 0

        # Process nn.Parameter objects as pseudo-modules
        for name, p in model.named_parameters(recurse=True):
            # Compute depth
            depth = name.count('.') + 1 if name else 0
            # Skip if depth exceeds recursive_depth (unless infinite)
            if recursive_depth != np.inf and depth > recursive_depth:
                continue

            p_id = id(p)
            if p_id not in param_ids:
                module_total = p.numel()
                module_trainable = p.numel() if p.requires_grad else 0
                module_percentage = 100 * module_trainable / module_total if module_total > 0 else 0.0
                indent = "  " * depth
                table_data.append([
                    depth,
                    indent + name,
                    f"{module_trainable:,}",
                    f"{module_total:,}",
                    f"{module_percentage:.4f}%"
                ])
                param_ids.add(p_id)
                total_trainable_in_table += module_trainable
                total_params_in_table += module_total

        # Process regular modules
        for name, module in model.named_modules():
            # Compute module depth
            depth = name.count('.') + 1 if name else 0
            # Skip if depth exceeds recursive_depth (unless infinite)
            if recursive_depth != np.inf and depth > recursive_depth:
                continue

            # Compute parameters for the current module
            module_trainable = 0
            module_total = 0
            module_param_ids = set()

            for _, p in module.named_parameters(recurse=False):
                p_id = id(p)
                if p_id not in param_ids:
                    module_param_ids.add(p_id)
                    module_total += p.numel()
                    if p.requires_grad:
                        module_trainable += p.numel()

            if module_total > 0:
                module_percentage = 100 * module_trainable / module_total if module_total > 0 else 0.0
                indent = "  " * depth
                table_data.append([
                    depth,
                    indent + (name or "root"),
                    f"{module_trainable:,}",
                    f"{module_total:,}",
                    f"{module_percentage:.4f}%"
                ])
                param_ids.update(module_param_ids)
                total_trainable_in_table += module_trainable
                total_params_in_table += module_total

        # Add totals row
        total_percentage = 100 * total_trainable_in_table / total_params_in_table if total_params_in_table > 0 else 0.0
        table_data.append([
            "-",  # No level for totals
            "Total",
            f"{total_trainable_in_table:,}",
            f"{total_params_in_table:,}",
            f"{total_percentage:.4f}%"
        ])

        # Create and log the table
        if table_data:
            headers = ["🔢 Level #", "🧩 Module", "📏 Trainable Params", "📊 Total Params", "📈 Trainable %"]
            table = tabulate(
                table_data,
                headers=headers,
                tablefmt="fancy_grid",
                stralign="left",
                numalign="right",
                floatfmt=".4f"
            )
            logging.info(f"\nParameter breakdown by module (depth <= {recursive_depth}):\n{table}")

    return trainable_params, total_params

def compute_trainable_params(model, recursive_depth=0, finegrained=False):
    """
    Compute the number of trainable and total parameters in the model.
    If recursive_depth > 0, generate a table of parameters for child modules up to the specified depth,
    including a Level # column and a totals row. If recursive_depth = np.inf, include all modules
    regardless of depth.
    
    Args:
        model (nn.Module): The PyTorch model to analyze.
        recursive_depth (int or float): Depth of recursion for child modules. Default is 0 (no recursion).
            Use np.inf to include all modules at any depth.
        finegrained (bool): If True, nn.Parameter objects are treated as pseudo-modules and listed separately.
            If False (default), only actual modules are listed.
    
    Returns:
        tuple: (trainable_params, total_params)
    """
    # Compute total trainable and all parameters for the entire model
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_percentage = 100 * trainable_params / total_params if total_params > 0 else 0.0

    # Log the summary for the entire model
    logging.info(
        f"📊 trainable params: {trainable_params:,} || all params: {total_params:,} || trainable%: {trainable_percentage:.4f}"
    )

    # If recursive_depth > 0, generate a table for child modules and optionally parameters
    if recursive_depth > 0:
        table_data = []
        param_ids = set()  # Track parameter IDs to avoid double-counting
        total_trainable_in_table = 0
        total_params_in_table = 0

        if finegrained:
            # First pass: Process all nn.Parameter objects as pseudo-modules
            for name, p in model.named_parameters(recurse=True):
                # Compute depth
                depth = name.count('.') + 1 if name else 0
                # Skip if depth exceeds recursive_depth (unless infinite)
                if recursive_depth != np.inf and depth > recursive_depth:
                    continue

                p_id = id(p)
                if p_id not in param_ids:
                    module_total = p.numel()
                    module_trainable = p.numel() if p.requires_grad else 0
                    module_percentage = 100 * module_trainable / module_total if module_total > 0 else 0.0
                    indent = "  " * depth
                    table_data.append([
                        depth,
                        indent + name,
                        f"{module_trainable:,}",
                        f"{module_total:,}",
                        f"{module_percentage:.4f}%"
                    ])
                    param_ids.add(p_id)
                    total_trainable_in_table += module_trainable
                    total_params_in_table += module_total

            # Second pass: Process modules that have parameters not yet counted
            for name, module in model.named_modules():
                # Compute module depth
                depth = name.count('.') + 1 if name else 0
                # Skip if depth exceeds recursive_depth (unless infinite)
                if recursive_depth != np.inf and depth > recursive_depth:
                    continue

                # Check if this module has any parameters that haven't been counted yet
                module_has_uncounted_params = False
                for _, p in module.named_parameters(recurse=False):
                    if id(p) not in param_ids:
                        module_has_uncounted_params = True
                        break

                # Only add module entry if it has uncounted parameters
                if module_has_uncounted_params:
                    module_trainable = 0
                    module_total = 0
                    module_param_ids = set()

                    for _, p in module.named_parameters(recurse=False):
                        p_id = id(p)
                        if p_id not in param_ids:
                            module_param_ids.add(p_id)
                            module_total += p.numel()
                            if p.requires_grad:
                                module_trainable += p.numel()

                    if module_total > 0:
                        module_percentage = 100 * module_trainable / module_total if module_total > 0 else 0.0
                        indent = "  " * depth
                        table_data.append([
                            depth,
                            indent + (name or "root"),
                            f"{module_trainable:,}",
                            f"{module_total:,}",
                            f"{module_percentage:.4f}%"
                        ])
                        param_ids.update(module_param_ids)
                        total_trainable_in_table += module_trainable
                        total_params_in_table += module_total

        else:
            # Original behavior: only process modules
            for name, module in model.named_modules():
                # Compute module depth
                depth = name.count('.') + 1 if name else 0
                # Skip if depth exceeds recursive_depth (unless infinite)
                if recursive_depth != np.inf and depth > recursive_depth:
                    continue

                # Compute parameters for the current module
                module_trainable = 0
                module_total = 0
                module_param_ids = set()

                # Only count parameters directly registered in the current module
                for _, p in module.named_parameters(recurse=False):
                    p_id = id(p)
                    if p_id not in param_ids:
                        module_param_ids.add(p_id)
                        module_total += p.numel()
                        if p.requires_grad:
                            module_trainable += p.numel()

                if module_total > 0:
                    module_percentage = 100 * module_trainable / module_total if module_total > 0 else 0.0
                    indent = "  " * depth
                    table_data.append([
                        depth,
                        indent + (name or "root"),
                        f"{module_trainable:,}",
                        f"{module_total:,}",
                        f"{module_percentage:.4f}%"
                    ])
                    param_ids.update(module_param_ids)
                    total_trainable_in_table += module_trainable
                    total_params_in_table += module_total

        # Sort table data by the module/parameter name to ensure consistent ordering
        # Extract the name without indentation for sorting
        def get_sort_key(row):
            name = row[1].strip()
            return name
        
        table_data.sort(key=get_sort_key)

        # Add totals row
        total_percentage = 100 * total_trainable_in_table / total_params_in_table if total_params_in_table > 0 else 0.0
        table_data.append([
            "-",  # No level for totals
            "Total",
            f"{total_trainable_in_table:,}",
            f"{total_params_in_table:,}",
            f"{total_percentage:.4f}%"
        ])

        # Create and log the table
        if table_data:
            headers = ["🔢 Level #", "🧩 Module", "📏 Trainable Params", "📊 Total Params", "📈 Trainable %"]
            table = tabulate(
                table_data,
                headers=headers,
                tablefmt="fancy_grid",
                stralign="left",
                numalign="right",
                floatfmt=".4f"
            )
            logging.info(f"\nParameter breakdown by module (depth <= {recursive_depth}):\n{table}")

    return trainable_params, total_params

In [ ]:
class TestModel(nn.Module):
    def __init__(self):
        super(TestModel, self).__init__()
        self.base_model = nn.ModuleDict({
            'label_embeddings': nn.Embedding(num_embeddings=1000, embedding_dim=128),
            'transformer': nn.ModuleDict({
                'linear': nn.Linear(128, 256),
                'lora_A': nn.Linear(128, 16, bias=False),
                'lora_B': nn.Linear(16, 256, bias=False),
            })
        })

        # Parameters added directly to the model
        self.bias_add = nn.Parameter(torch.zeros(5, 128))
        self.boost_add = nn.Parameter(torch.zeros(10, 128))

        for p in self.base_model['transformer']['linear'].parameters():
            p.requires_grad = False

        self.pooling = nn.AdaptiveAvgPool1d(output_size=32)
        self.classifier = nn.Linear(32, 1)

    def forward(self, x):
        emb = self.base_model['label_embeddings'](x)
        trans = self.base_model['transformer']['linear'](emb)
        lora = self.base_model['transformer']['lora_B'](
            self.base_model['transformer']['lora_A'](emb)
        )
        out = trans + lora + self.boost_add.unsqueeze(0) + self.bias_add.unsqueeze(0)
        pooled = self.pooling(out.permute(0, 2, 1)).permute(0, 2, 1)
        return self.classifier(pooled)

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Instantiate the test model
model = TestModel()

# Perform testing
for depth in range(4):
    logging.info(f"\nTesting recursive_depth={depth}:\n")
    compute_trainable_params(model, recursive_depth=depth)
    logging.info("-"*100)

logging.info("\nTesting recursive_depth=np.inf:")
compute_trainable_params(model, recursive_depth=np.inf)

In [ ]:
#| export
def unfreeze_layers(model, pct_to_unfreeze):
    """
    Progressively unfreezes layers from the end of the model until the specified
    percentage of eligible (floating-point/complex) model parameters is trainable.
    Already unfrozen layers remain unfrozen, and if the model is already unfrozen to
    the specified percentage, no changes are made. Only floating-point and complex
    tensors are set to require gradients.
    
    Args:
        model: The PyTorch model (e.g., Hugging Face transformer model).
        pct_to_unfreeze (float): Percentage of eligible parameters to unfreeze (0.0 to 1.0).
    
    Returns:
        None
    """
    # Get the list of transformer layers
    try:
        layers = model.model.layers
    except AttributeError:
        logging.error("🚫 Model does not have 'model.layers'. Check model structure.")
        raise AttributeError("Model does not have 'model.layers'.")
    
    total_layers = len(layers)
    
    # Validate input
    if not 0.0 <= pct_to_unfreeze <= 1.0:
        logging.error("🚫 pct_to_unfreeze must be between 0.0 and 1.0, got %.2f.", pct_to_unfreeze)
        raise ValueError("pct_to_unfreeze must be between 0.0 and 1.0.")
    
    # Calculate total eligible parameters (floating-point/complex only)
    eligible_params = sum(p.numel() for p in model.parameters() if p.dtype in (torch.float16, torch.float32, torch.float64, torch.complex64, torch.complex128))
    if eligible_params == 0:
        logging.error("🚫 No eligible (floating-point/complex) parameters found in model.")
        raise ValueError("No eligible parameters found.")
    
    # Calculate target number of trainable parameters
    target_trainable_params = math.ceil(eligible_params * pct_to_unfreeze)
    
    # Count currently trainable parameters
    current_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad and p.dtype in (torch.float16, torch.float32, torch.float64, torch.complex64, torch.complex128))
    
    # Check if already unfrozen to or beyond the requested percentage
    if current_trainable_params >= target_trainable_params:
        logging.info("✅ Model already has %d/%d eligible parameters trainable (requested %d for %.1f%%). No changes made.",
                     current_trainable_params, eligible_params, target_trainable_params, pct_to_unfreeze * 100)
        return
    
    # Calculate remaining parameters needed to reach target
    params_needed = target_trainable_params - current_trainable_params
    
    # Iterate over layers from the end to calculate cumulative eligible parameters
    cumulative_params = 0
    layers_to_unfreeze = []
    for layer_idx in range(total_layers - 1, -1, -1):  # Start from last layer
        layer = layers[layer_idx]
        # Count eligible parameters in this layer
        layer_params = sum(p.numel() for p in layer.parameters() if p.dtype in (torch.float16, torch.float32, torch.float64, torch.complex64, torch.complex128))
        if layer_params > 0:  # Only include layers with eligible parameters
            cumulative_params += layer_params
            layers_to_unfreeze.append(layer_idx)
            if cumulative_params >= params_needed:
                break
    
    # Unfreeze the selected layers
    num_layers_to_unfreeze = len(layers_to_unfreeze)
    if num_layers_to_unfreeze == 0:
        logging.info("⚠️ No additional layers with eligible parameters found to reach %.1f%% trainable parameters.", pct_to_unfreeze * 100)
        return
    
    logging.info("🔄 Unfreezing %d layers (from layer %d to %d) to reach %.1f%% trainable parameters (%d/%d eligible).",
                 num_layers_to_unfreeze, min(layers_to_unfreeze), max(layers_to_unfreeze), pct_to_unfreeze * 100,
                 cumulative_params + current_trainable_params, eligible_params)
    
    for layer_idx in layers_to_unfreeze:
        layer = layers[layer_idx]
        for param in layer.parameters():
            # Only set requires_grad for floating-point or complex tensors
            if param.dtype in (torch.float16, torch.float32, torch.float64, torch.complex64, torch.complex128):
                param.requires_grad = True
            else:
                logging.debug("ℹ️ Skipping requires_grad for non-floating-point tensor (dtype: %s) in layer %d.", param.dtype, layer_idx)
    
    # Log final status
    final_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad and p.dtype in (torch.float16, torch.float32, torch.float64, torch.complex64, torch.complex128))
    final_frozen_params = eligible_params - final_trainable_params
    logging.info("📊 Final status: %d parameters frozen ❄️, %d parameters unfrozen 🔥 (%.1f%% of %d eligible).",
                 final_frozen_params, final_trainable_params, (final_trainable_params / eligible_params) * 100, eligible_params)

## Git Utilities

In [ ]:
#| export
import subprocess
import json
from datetime import datetime
from typing import Dict, Optional, List, Tuple
from tabulate import tabulate

In [ ]:
#| export
import subprocess
import json
from datetime import datetime
from typing import Dict, Optional, List, Tuple
from tabulate import tabulate

def get_git_file_status() -> Dict[str, List[str]]:
    """
    Get detailed file status information from git.
    
    Returns:
        Dictionary with lists of files in different states
    """
    try:
        # Get git status in porcelain format for parsing
        status_result = subprocess.run(['git', 'status', '--porcelain'], 
                                     capture_output=True, text=True, check=True)
        status_lines = status_result.stdout.strip().split('\n')
        
        # Initialize file state collections
        file_states = {
            'staged_new': [],      # A  - new file staged
            'staged_modified': [], # M  - modified file staged
            'staged_deleted': [],  # D  - deleted file staged
            'staged_renamed': [],  # R  - renamed file staged
            'modified': [],        # _M - modified but not staged
            'deleted': [],         # _D - deleted but not staged
            'untracked': [],       # ?? - untracked files
            'conflicted': [],      # UU, AA, etc. - merge conflicts
            'ignored': []          # !! - ignored files (if shown)
        }
        
        if not status_lines or status_lines == ['']:
            return file_states
        
        for line in status_lines:
            if len(line) < 3:
                continue
                
            index_status = line[0]  # Staged area status
            worktree_status = line[1]  # Working directory status
            filename = line[3:]  # Skip the space and get filename
            
            # Parse different file states
            if line.startswith('??'):
                file_states['untracked'].append(filename)
            elif line.startswith('!!'):
                file_states['ignored'].append(filename)
            elif index_status in ['U'] or worktree_status in ['U'] or line.startswith('AA') or line.startswith('UU'):
                file_states['conflicted'].append(filename)
            else:
                # Check staged area (index) status
                if index_status == 'A':
                    file_states['staged_new'].append(filename)
                elif index_status == 'M':
                    file_states['staged_modified'].append(filename)
                elif index_status == 'D':
                    file_states['staged_deleted'].append(filename)
                elif index_status == 'R':
                    file_states['staged_renamed'].append(filename)
                
                # Check working directory status
                if worktree_status == 'M':
                    file_states['modified'].append(filename)
                elif worktree_status == 'D':
                    file_states['deleted'].append(filename)
        
        return file_states
        
    except subprocess.CalledProcessError:
        return {}

def get_git_repo_stats() -> Optional[Dict[str, str]]:
    """
    Get current git repository statistics including commit hash, message, 
    timestamp, and author information.
    
    Returns:
        Dictionary containing git stats or None if not a git repository
    """
    try:
        # Check if we're in a git repository
        subprocess.run(['git', 'rev-parse', '--git-dir'], 
                      capture_output=True, check=True)
        
        # Get commit hash
        hash_result = subprocess.run(['git', 'rev-parse', 'HEAD'], 
                                   capture_output=True, text=True, check=True)
        commit_hash = hash_result.stdout.strip()
        
        # Get commit message
        msg_result = subprocess.run(['git', 'log', '-1', '--pretty=format:%s'], 
                                  capture_output=True, text=True, check=True)
        commit_message = msg_result.stdout.strip()
        
        # Get commit timestamp
        time_result = subprocess.run(['git', 'log', '-1', '--pretty=format:%ci'], 
                                   capture_output=True, text=True, check=True)
        commit_time = time_result.stdout.strip()
        
        # Get author name and email
        author_result = subprocess.run(['git', 'log', '-1', '--pretty=format:%an <%ae>'], 
                                     capture_output=True, text=True, check=True)
        author = author_result.stdout.strip()
        
        # Get branch name
        branch_result = subprocess.run(['git', 'branch', '--show-current'], 
                                     capture_output=True, text=True, check=True)
        branch = branch_result.stdout.strip()
        
        # Get repository name (from remote URL or directory name)
        try:
            repo_result = subprocess.run(['git', 'remote', 'get-url', 'origin'], 
                                       capture_output=True, text=True, check=True)
            repo_url = repo_result.stdout.strip()
            # Extract repo name from URL
            repo_name = repo_url.split('/')[-1].replace('.git', '')
        except subprocess.CalledProcessError:
            # Fallback to directory name
            import os
            repo_name = os.path.basename(os.getcwd())
        
        # Get number of commits on current branch
        try:
            commit_count_result = subprocess.run(['git', 'rev-list', '--count', 'HEAD'], 
                                               capture_output=True, text=True, check=True)
            commit_count = commit_count_result.stdout.strip()
        except subprocess.CalledProcessError:
            commit_count = "Unknown"
        
        # Get file status information
        file_states = get_git_file_status()
        
        return {
            'repo_name': repo_name,
            'commit_hash': commit_hash,
            'short_hash': commit_hash[:7],
            'commit_message': commit_message,
            'commit_time': commit_time,
            'author': author,
            'branch': branch,
            'commit_count': commit_count,
            'file_states': file_states
        }
        
    except subprocess.CalledProcessError:
        print("❌ Error: Not a git repository or git command failed")
        return None
    except FileNotFoundError:
        print("❌ Error: Git is not installed or not in PATH")
        return None

def get_working_directory_summary(file_states: Dict[str, List[str]], include_untracked=False) -> str:
    """
    Create a summary of working directory status with emojis.
    """
    total_staged = (len(file_states['staged_new']) + 
                   len(file_states['staged_modified']) + 
                   len(file_states['staged_deleted']) + 
                   len(file_states['staged_renamed']))
    
    total_unstaged = (len(file_states['modified']) + 
                     len(file_states['deleted']))
    
    total_untracked = len(file_states['untracked']) if include_untracked else 0
    total_conflicted = len(file_states['conflicted'])
    
    if total_conflicted > 0:
        return f"🔥 CONFLICTS ({total_conflicted})"
    elif total_staged > 0 and (total_unstaged > 0 or total_untracked > 0):
        # Mixed state: staged + (unstaged or untracked)
        unstaged_part = f"{total_unstaged} unstaged" if total_unstaged > 0 else ""
        untracked_part = f"{total_untracked} untracked" if total_untracked > 0 else ""
        
        if unstaged_part and untracked_part:
            return f"⚡ MIXED ({total_staged} staged, {unstaged_part}, {untracked_part})"
        elif unstaged_part:
            return f"⚡ MIXED ({total_staged} staged, {unstaged_part})"
        else:
            return f"⚡ MIXED ({total_staged} staged, {untracked_part})"
    elif total_staged > 0:
        return f"🟢 STAGED ({total_staged})"
    elif total_unstaged > 0 or total_untracked > 0:
        return f"🟡 CHANGES ({total_unstaged + total_untracked})"
    else:
        return "✅ CLEAN"

def print_git_stats(table_format="grid", show_files=True, show_untracked=False):
    """
    Print git repository statistics in a beautifully formatted table.
    
    Args:
        table_format: Format for tabulate (grid, fancy_grid, simple, etc.)
        show_files: Whether to show detailed file status
    """
    stats = get_git_repo_stats()
    
    if not stats:
        print("❌ Could not retrieve git repository information")
        return
    
    file_states = stats['file_states']
    wd_summary = get_working_directory_summary(file_states, include_untracked=show_untracked)
    
    # Create main info table
    table_data = [
        ["📁 Repository", stats['repo_name']],
        ["🌿 Branch", stats['branch']],
        ["🔍 Commit Hash", f"{stats['short_hash']} ({stats['commit_hash']})"],
        ["💬 Last Message", stats['commit_message']],
        ["👤 Author", stats['author']],
        ["📅 Date & Time", stats['commit_time']],
        ["📊 Total Commits", stats['commit_count']],
        ["📋 Working Directory", wd_summary],
        ["🔬 Inspect Commit", f"git show {stats['short_hash']} --stat"],
        ["🕰️ Checkout Commit", f"git checkout {stats['short_hash']}"]
    ]
    
    # Print header
    print("\n" + "🚀 Git Repository Statistics 🚀".center(80))
    print("=" * 80)
    
    # Print main table
    print(tabulate(table_data, 
                  headers=["📋 Property", "📝 Value"], 
                  tablefmt=table_format,
                  colalign=("left", "left")))
    
    if show_files:
        print_file_status_details(file_states, table_format, show_untracked)

def print_file_status_details(file_states: Dict[str, List[str]], table_format="grid", show_untracked=False):
    """
    Print detailed file status information.
    """
    print(f"\n" + "📂 File Status Details 📂".center(80))
    print("=" * 80)
    
    # Create file status summary table
    status_summary = []
    
    # Staged files
    staged_count = len(file_states['staged_new']) + len(file_states['staged_modified']) + \
                  len(file_states['staged_deleted']) + len(file_states['staged_renamed'])
    if staged_count > 0:
        status_summary.append(["🟢 Staged Area", f"{staged_count} files ready to commit"])
    
    # Modified files
    if file_states['modified']:
        status_summary.append(["🟡 Modified", f"{len(file_states['modified'])} files changed but not staged"])
    
    # Deleted files
    if file_states['deleted']:
        status_summary.append(["🗑️ Deleted", f"{len(file_states['deleted'])} files deleted but not staged"])
    
    # Untracked files (only if requested)
    if file_states['untracked'] and show_untracked:
        status_summary.append(["❓ Untracked", f"{len(file_states['untracked'])} new files not in git"])
    
    # Conflicted files
    if file_states['conflicted']:
        status_summary.append(["🔥 Conflicts", f"{len(file_states['conflicted'])} files with merge conflicts"])
    
    if status_summary:
        print(tabulate(status_summary, 
                      headers=["🏷️ Status", "📄 Description"], 
                      tablefmt=table_format))
    else:
        print("✅ Working directory is clean - no changes to commit")
    
    # Show individual files if there are any changes
    if any(files for files in file_states.values()):
        print(f"\n" + "📋 Individual Files 📋".center(60))
        print("-" * 60)
        
        file_details = []
        
        # Add staged files
        for filename in file_states['staged_new']:
            file_details.append(["🆕 New (Staged)", filename])
        for filename in file_states['staged_modified']:
            file_details.append(["✏️ Modified (Staged)", filename])
        for filename in file_states['staged_deleted']:
            file_details.append(["🗑️ Deleted (Staged)", filename])
        for filename in file_states['staged_renamed']:
            file_details.append(["📝 Renamed (Staged)", filename])
        
        # Add unstaged files
        for filename in file_states['modified']:
            file_details.append(["⚠️ Modified", filename])
        for filename in file_states['deleted']:
            file_details.append(["❌ Deleted", filename])
        
        # Add untracked files (only if requested)
        if show_untracked:
            for filename in file_states['untracked']:
                file_details.append(["❓ Untracked", filename])
        for filename in file_states['conflicted']:
            file_details.append(["🔥 Conflict", filename])
        
        if file_details:
            # Limit to first 20 files to avoid overwhelming output
            if len(file_details) > 20:
                file_details = file_details[:20]
                file_details.append(["...", f"and {len(file_details) - 20} more files"])
            
            print(tabulate(file_details, 
                          headers=["🏷️ Status", "📄 Filename"], 
                          tablefmt=table_format))

def get_compact_git_info(include_untracked=False) -> str:
    """
    Get a compact one-line git info string with emojis.
    
    Args:
        include_untracked: Whether to include untracked files in status summary
    
    Returns:
        Formatted string with key git info
    """
    stats = get_git_repo_stats()
    if not stats:
        return "❌ Not a git repository"
    
    wd_summary = get_working_directory_summary(stats['file_states'], include_untracked=include_untracked)
    return (f"📁 {stats['repo_name']} | "
            f"🌿 {stats['branch']} | "
            f"🔍 {stats['short_hash']} | "
            f"👤 {stats['author'].split('<')[0].strip()} | "
            f"{wd_summary} | "
            f"🔬 git show {stats['short_hash']}")

# Example usage
if __name__ == "__main__":
    # Full detailed view (excludes untracked files by default)
    print_git_stats()
    
    # Include untracked files
    print(f"\n" + "📂 With Untracked Files 📂".center(60))
    print_git_stats(show_untracked=True)
    
    # Compact one-liner
    print(f"\n🚀 Quick Info: {get_compact_git_info()}")

    # Compact one-liner
    print(f"\n🚀 Quick Info: {get_compact_git_info(include_untracked=True)}")
    
    # Just basic info without file details
    print(f"\n" + "📊 Basic Info Only 📊".center(60))
    print_git_stats(show_files=False)


                         🚀 Git Repository Statistics 🚀                          
+----------------------+-------------------------------------------------------------------+
| 📋 Property          | 📝 Value                                                          |
+======================+===================================================================+
| 📁 Repository        | xcube                                                             |
+----------------------+-------------------------------------------------------------------+
| 🌿 Branch            | plant                                                             |
+----------------------+-------------------------------------------------------------------+
| 🔍 Commit Hash       | 90f8d89 (90f8d8947f9e984a8a1fdc3c111c1b6dfb210ea4)                |
+----------------------+-------------------------------------------------------------------+
| 💬 Last Message      | working on importing pretrained l2r model in multi-label class

## Export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()